# Import Qwen

In [1]:
%load_ext autoreload

%autoreload 2

import warnings

# Ignore all FutureWarnings to clean up your notebook outputs
warnings.filterwarnings("ignore", category=FutureWarning)

from langchain_core.messages import HumanMessage, ToolMessage
from IPython.display import Image, display
from app.graph.workflow import build_graph
from app.core.app import container
from pprint import pprint as ppr
from app.memory.extractor import MemoryExtractor

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/nguyen/micromamba/envs/unsloth_fresh/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


ModuleNotFoundError: No module named 'app.retrieval.service'

# Test Graph Workflow

In [ ]:
graph = build_graph()

result = graph.invoke(
    {
        "messages": [
            HumanMessage(
                content="I'm looking for a job, you know it's about AI engineer or ai researcher role" \
                "But I'm quite concern about when will the recruiter response, I mean do they take " \
                "many days? I worry that they won't contact me. Anyway, do you know what should I response" \
                "when they ask me what is LangChain?",
            )
        ]
    }
)

display(Image(graph.get_graph().draw_mermaid_png()))

ppr(result)

# Test Memory

In [ ]:
extracted_memories = container.memory_extractor.extract(result["messages"])

In [ ]:
extracted_memories

In [ ]:
# Save memories to table "memories"
container.memory_manager.memory_store.save(extracted_memories)

In [ ]:
faiss_items = []

# Save mapping of memory and faiss_id to table "memory_faiss_mapping"
for memory in extracted_memories:

    # Get faiss_id
    faiss_id = (
        container.memory_manager.memory_store
        .get_next_faiss_id()
    )

    # Insert the mapping
    container.memory_manager.memory_store.save_faiss_mapping(
        faiss_id=faiss_id,
        memory_id=memory.id,
    )

    # Collect list of faiss id paired with is content
    faiss_items.append(
        (
            faiss_id,
            memory.content,
        )
    )

# Add all faiss items to the index
container.faiss_store.add_many(
    faiss_items
)

container.faiss_store.save()